### Links and Resources
| Method / Command | Description |
|------------------|-------------|
| [DeltaTable.history()](https://docs.delta.io/latest/api/python/spark/index.html) | Returns the full commit history as a DataFrame |
| [DESCRIBE HISTORY (SQL)](https://docs.databricks.com/en/sql/language-manual/delta-describe-history.html) | SQL equivalent of `history()` — supports `LIMIT` |
| [VERSION AS OF](https://docs.databricks.com/en/delta/history.html#delta-time-travel-syntax) | Read the table as it was at a specific version number |
| [TIMESTAMP AS OF](https://docs.databricks.com/en/delta/history.html#delta-time-travel-syntax) | Read the table as it was at a specific point in time |
| [RESTORE TABLE](https://docs.databricks.com/en/sql/language-manual/delta-restore.html) | Revert a Delta table to an earlier version (adds a new commit) |
| [operationMetrics](https://docs.databricks.com/en/delta/history.html#operation-metrics-keys) | Per-operation row and file counters available in every history entry |
| [delta.deletedFileRetentionDuration](https://docs.databricks.com/en/delta/history.html#configure-data-retention) | Controls how far back time travel is possible (default 7 days) |

In [ ]:
import os, sys

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(
            f"Wrong Python kernel — 'databricks.connect' not found in:\n  {sys.executable}\n\n"
            "In VSCode: click the kernel selector (top-right of notebook) "
            "and choose the interpreter at:\n"
            r"  C:\Source\Dsa\DataBricks\.venv\Scripts\python.exe"
        )
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()


def display(df, n=20):
    df.show(n, truncate=False)

# History and Time Travel

Delta Lake records every mutation in its `_delta_log/`. This gives you two powerful capabilities:

- **History** — inspect the full audit trail: who did what, when, with which predicate, and how many rows were affected.
- **Time Travel** — read the table exactly as it was at any past version or timestamp, without maintaining separate snapshots.

### Three time-travel syntaxes

| Syntax | DataFrame API | SQL |
|--------|--------------|-----|
| By version number | `.option("versionAsOf", 3)` | `VERSION AS OF 3` |
| By timestamp | `.option("timestampAsOf", "2025-01-15")` | `TIMESTAMP AS OF '2025-01-15'` |
| Short form (version) | `.load("path@v3")` | `table@v3` |

All three read from the same `_delta_log/` — they differ only in how you specify which version you want.

## Setup — Build a Table with a Rich History

A richer history makes the examples more meaningful.  
The following cell applies 8 operations across different types so every operation appears in the audit trail.

In [ ]:
from pyspark.sql import Row, functions as F
from delta.tables import DeltaTable
import time

TABLE = "population_metrics.default.product_inventory"

initial_rows = [
    Row(product_id=1,  name="Laptop Pro 15",     category="Electronics", price=1299.99, in_stock=True),
    Row(product_id=2,  name="Wireless Mouse",     category="Electronics", price=25.49,  in_stock=True),
    Row(product_id=3,  name="USB-C Hub",          category="Electronics", price=39.99,  in_stock=False),
    Row(product_id=4,  name="Standing Desk",      category="Furniture",   price=499.00, in_stock=True),
    Row(product_id=5,  name="Ergonomic Chair",    category="Furniture",   price=349.00, in_stock=True),
    Row(product_id=6,  name="Monitor Arm",        category="Furniture",   price=89.95,  in_stock=False),
    Row(product_id=7,  name="Notebook (A5)",      category="Stationery",  price=4.99,   in_stock=True),
    Row(product_id=8,  name="Ballpoint Pen 10pk", category="Stationery",  price=2.49,   in_stock=True),
    Row(product_id=9,  name="Sticky Notes 5pk",   category="Stationery",  price=3.79,   in_stock=True),
    Row(product_id=10, name="Whiteboard Marker",  category="Stationery",  price=5.99,   in_stock=False),
]

# Version 0 — CREATE OR REPLACE TABLE AS SELECT
spark.createDataFrame(initial_rows).write.format("delta").mode("overwrite").saveAsTable(TABLE)
time.sleep(2)   # ensure distinct timestamps between commits

dt = DeltaTable.forName(spark, TABLE)

# Version 1 — APPEND two new products
spark.createDataFrame([
    Row(product_id=11, name="Coffee Mug",   category="Kitchen",     price=12.00, in_stock=True),
    Row(product_id=12, name="Desk Lamp LED",category="Electronics", price=45.00, in_stock=True),
]).write.format("delta").mode("append").saveAsTable(TABLE)
time.sleep(2)

# Version 2 — UPDATE: 10% price increase on Furniture
dt = DeltaTable.forName(spark, TABLE)
dt.update("category = 'Furniture'", {"price": F.round(F.col("price") * 1.10, 2)})
time.sleep(2)

# Version 3 — UPDATE: restock out-of-stock items
dt = DeltaTable.forName(spark, TABLE)
dt.update(F.col("in_stock") == False, {"in_stock": F.lit(True)})
time.sleep(2)

# Version 4 — DELETE: remove all Kitchen products
dt = DeltaTable.forName(spark, TABLE)
dt.delete("category = 'Kitchen'")
time.sleep(2)

# Version 5 — MERGE: upsert a pricing correction + one new product
corrections = spark.createDataFrame([
    Row(product_id=1,  name="Laptop Pro 15",   category="Electronics", price=1099.99, in_stock=True),
    Row(product_id=13, name="Smart Speaker",   category="Electronics", price=79.00,   in_stock=True),
])
dt = DeltaTable.forName(spark, TABLE)
(
    dt.merge(corrections, "target.product_id = source.product_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)
time.sleep(2)

# Version 6 — UPDATE: accidental price zeroing (we will restore from this later)
dt = DeltaTable.forName(spark, TABLE)
dt.update("category = 'Electronics'", {"price": F.lit(0.0)})
time.sleep(2)

# Version 7 — OPTIMIZE
spark.sql(f"OPTIMIZE {TABLE}")

print("History built — 8 versions (0–7) across CREATE, APPEND, UPDATE, DELETE, MERGE, OPTIMIZE.")
print("\nCurrent table state:")
display(spark.table(TABLE).orderBy("product_id"))

# Part 1 — `history()` — The Full Audit Trail

`DeltaTable.history(n)` returns a DataFrame with one row per commit, newest-first.  
Passing `n` limits to the last `n` versions; omitting it returns everything.

In [ ]:
dt = DeltaTable.forName(spark, TABLE)

# Full history, oldest-first for readability
history_df = dt.history().orderBy("version")

print(f"Total versions: {history_df.count()}")
print("\nAll columns available in history():")
print([f.name for f in history_df.schema.fields])

### `history()` Column Reference

| Column | Type | Contents |
|--------|------|----------|
| `version` | Long | Monotonically increasing commit number |
| `timestamp` | Timestamp | When the commit was written |
| `userId` / `userName` | String | Identity of the writer |
| `operation` | String | `CREATE OR REPLACE TABLE AS SELECT`, `WRITE`, `UPDATE`, `DELETE`, `MERGE`, `OPTIMIZE`, `RESTORE`, … |
| `operationParameters` | Map | Predicate, partition info, Z-ORDER columns, etc. — varies by operation |
| `operationMetrics` | Map | Row and file counts — varies by operation (see Part 2) |
| `job` | Struct | Databricks Job info if the commit was made by a Job |
| `notebook` | Struct | Notebook path/ID if the commit was made from a notebook |
| `clusterId` | String | Cluster that wrote the commit |
| `readVersion` | Long | Version the writer read before committing (for optimistic concurrency) |
| `isolationLevel` | String | Transaction isolation (`WriteSerializable`, `Serializable`) |
| `isBlindAppend` | Boolean | `true` if no existing data was read or modified (pure INSERT) |
| `engineInfo` | String | Delta version and Spark version that wrote the commit |

In [ ]:
# Key columns for auditing
display(
    history_df.select(
        "version", "timestamp", "operation",
        "operationParameters", "isBlindAppend", "readVersion"
    )
)

### `isBlindAppend` — Concurrency Signal

`isBlindAppend = true` means the writer added new files without reading or modifying any existing data.  
Concurrent writers can safely proceed alongside a blind append with no conflict risk.  

Pure `INSERT` / `APPEND` writes set `isBlindAppend = true`.  
`UPDATE`, `DELETE`, `MERGE`, and `OPTIMIZE` set it to `false` because they read existing files.

### SQL Equivalent — `DESCRIBE HISTORY`

`DESCRIBE HISTORY` is the SQL form of `history()`. It supports an optional `LIMIT` clause.

In [ ]:
# DESCRIBE HISTORY — SQL form, supports LIMIT
display(
    spark.sql(f"DESCRIBE HISTORY {TABLE} LIMIT 5")
    .select("version", "timestamp", "operation", "operationParameters")
    .orderBy("version")
)

# Part 2 — `operationMetrics` by Operation Type

Every commit stores a `operationMetrics` map with counters that are specific to the operation that produced it.  
Querying these metrics is the fastest way to answer: *how much did this commit actually do?*

In [ ]:
dt = DeltaTable.forName(spark, TABLE)
history_df = dt.history().orderBy("version")

# Helper: extract the most useful metrics from each operation type
metrics_df = history_df.select(
    "version",
    "operation",
    # WRITE / APPEND
    F.col("operationMetrics")["numFiles"].alias("numFiles"),
    F.col("operationMetrics")["numOutputRows"].alias("outputRows"),
    # UPDATE
    F.col("operationMetrics")["numUpdatedRows"].alias("updatedRows"),
    F.col("operationMetrics")["numCopiedRows"].alias("copiedRows"),
    # DELETE
    F.col("operationMetrics")["numDeletedRows"].alias("deletedRows"),
    # MERGE
    F.col("operationMetrics")["numTargetRowsInserted"].alias("mergeInserted"),
    F.col("operationMetrics")["numTargetRowsUpdated"].alias("mergeUpdated"),
    F.col("operationMetrics")["numTargetRowsDeleted"].alias("mergeDeleted"),
    # OPTIMIZE
    F.col("operationMetrics")["numFilesAdded"].alias("filesAdded"),
    F.col("operationMetrics")["numFilesRemoved"].alias("filesRemoved"),
)

print("operationMetrics per version (null = not applicable for that operation type):")
display(metrics_df)

### `operationMetrics` Keys by Operation

| Operation | Key metrics |
|-----------|------------|
| `WRITE` / `APPEND` | `numFiles`, `numOutputRows`, `numOutputBytes` |
| `UPDATE` | `numUpdatedRows`, `numCopiedRows`, `numAddedFiles`, `numRemovedFiles` |
| `DELETE` | `numDeletedRows`, `numAddedFiles`, `numRemovedFiles` |
| `MERGE` | `numSourceRows`, `numTargetRowsInserted`, `numTargetRowsUpdated`, `numTargetRowsDeleted`, `numTargetRowsCopied` |
| `OPTIMIZE` | `numFilesAdded`, `numFilesRemoved`, `numDeletionVectorsRemoved`, `filesAdded.avg` |
| `RESTORE` | `numRestoredFiles`, `removedFilesSize`, `numRemovedFiles` |

# Part 3 — Time Travel by Version: `VERSION AS OF`

The simplest form of time travel: specify the exact version number you want to read.  
Version numbers are stable — version 2 will always be version 2 regardless of what happens later.

In [ ]:
# --- Syntax 1: SQL string inside spark.sql() ---
df_v0 = spark.sql(f"SELECT * FROM {TABLE} VERSION AS OF 0 ORDER BY product_id")
print(f"Version 0 — original table ({df_v0.count()} rows):")
display(df_v0)

In [ ]:
# --- Syntax 2: DataFrame API with .option() ---
# Requires the physical storage location — use detail() to retrieve it
location = DeltaTable.forName(spark, TABLE).detail().collect()[0]["location"]

df_v2 = (
    spark.read.format("delta")
    .option("versionAsOf", 2)
    .load(location)
    .orderBy("product_id")
)
print(f"Version 2 — after Furniture price increase ({df_v2.count()} rows):")
display(df_v2.filter("category = 'Furniture'"))

In [ ]:
# --- Syntax 3: @v shorthand (Databricks SQL extension) ---
# Works in SQL cells and spark.sql() for Unity Catalog tables
df_v5 = spark.sql(f"SELECT * FROM {TABLE}@v5 ORDER BY product_id")
print(f"Version 5 — after MERGE ({df_v5.count()} rows):")
display(df_v5)

In [ ]:
# @v shorthand also works in the path for Delta tables loaded by path
df_v1_path = spark.read.format("delta").load(f"{location}@v1").orderBy("product_id")
print(f"Version 1 via path@v1 shorthand ({df_v1_path.count()} rows):")
display(df_v1_path)

### Row Count Across All Versions

A quick scan of how the table size changed with each operation.

In [ ]:
dt = DeltaTable.forName(spark, TABLE)
max_version = dt.history(1).collect()[0]["version"]

print("Row count at each version:")
for v in range(max_version + 1):
    count = spark.sql(f"SELECT COUNT(*) FROM {TABLE} VERSION AS OF {v}").collect()[0][0]
    op    = dt.history().filter(f"version = {v}").collect()[0]["operation"]
    print(f"  v{v:2d}  {op:45s}  {count:3d} rows")

# Part 4 — Time Travel by Timestamp: `TIMESTAMP AS OF`

Version numbers require knowing the log. Timestamps let you say *"give me the table as it was at 9 AM yesterday"* without knowing the version number.  
Delta finds the latest commit whose timestamp is **≤** the requested timestamp.

Accepted timestamp formats:
- `'2025-03-20'` — date only (time defaults to `00:00:00`)
- `'2025-03-20 09:15:00'` — date and time
- `'2025-03-20T09:15:00.000+00:00'` — ISO 8601 with timezone

In [ ]:
# Retrieve the exact timestamps of each commit from history
dt = DeltaTable.forName(spark, TABLE)
history_rows = dt.history().orderBy("version").select("version", "timestamp", "operation").collect()

print("Commit timestamps (use these in TIMESTAMP AS OF queries):")
for row in history_rows:
    print(f"  v{row['version']:2d}  {str(row['timestamp']):30s}  {row['operation']}")

In [ ]:
# Pick the timestamp of a specific version to time-travel to
# We'll use the timestamp of version 3 (after restocking)
v3_timestamp = str(history_rows[3]["timestamp"])

# --- Syntax 1: SQL TIMESTAMP AS OF ---
df_ts = spark.sql(f"SELECT * FROM {TABLE} TIMESTAMP AS OF '{v3_timestamp}' ORDER BY product_id")
print(f"TIMESTAMP AS OF '{v3_timestamp}' — should match VERSION AS OF 3:")
display(df_ts)

In [ ]:
# --- Syntax 2: DataFrame API with .option("timestampAsOf") ---
df_ts_api = (
    spark.read.format("delta")
    .option("timestampAsOf", v3_timestamp)
    .load(location)
    .orderBy("product_id")
)
print(f"DataFrame API timestampAsOf — row count: {df_ts_api.count()}")
display(df_ts_api)

### `TIMESTAMP AS OF` Resolution Rules

Delta resolves a timestamp to the **latest version whose commit timestamp ≤ requested timestamp**.

```
Commits:  v0@09:00  v1@09:05  v2@09:12  v3@09:20  v4@09:31

TIMESTAMP AS OF '09:10'  →  resolves to v1  (latest commit ≤ 09:10)
TIMESTAMP AS OF '09:20'  →  resolves to v3  (exact match)
TIMESTAMP AS OF '08:59'  →  ERROR — before the table was created
```

If the requested timestamp predates the oldest available commit an error is raised.

# Part 5 — Practical Audit Patterns

## 5a — Track How a Specific Row Changed Over Time

Read every version and union them together with a `_version` marker column to build a changelog for one row.

In [ ]:
dt = DeltaTable.forName(spark, TABLE)
max_version = dt.history(1).collect()[0]["version"]

# Build a changelog for product_id = 1 across all versions
changelog_frames = []
for v in range(max_version + 1):
    frame = (
        spark.sql(f"SELECT * FROM {TABLE} VERSION AS OF {v}")
        .filter("product_id = 1")
        .withColumn("_version", F.lit(v))
    )
    changelog_frames.append(frame)

changelog = changelog_frames[0]
for f in changelog_frames[1:]:
    changelog = changelog.unionByName(f, allowMissingColumns=True)

print("Changelog for product_id = 1 (price and in_stock across all versions):")
display(changelog.select("_version", "product_id", "name", "price", "in_stock").orderBy("_version"))

## 5b — Diff Two Versions: What Changed Between v2 and v5?

Join two snapshots on the primary key and compare column values to find rows that were added, removed, or modified.

In [ ]:
snap_v2 = spark.sql(f"SELECT * FROM {TABLE} VERSION AS OF 2").alias("v2")
snap_v5 = spark.sql(f"SELECT * FROM {TABLE} VERSION AS OF 5").alias("v5")

# Rows present in v5 but not in v2 (net-new rows)
added = snap_v5.join(snap_v2, on="product_id", how="left_anti").withColumn("change", F.lit("ADDED"))

# Rows present in v2 but gone from v5 (deleted rows)
removed = snap_v2.join(snap_v5, on="product_id", how="left_anti").withColumn("change", F.lit("REMOVED"))

# Rows in both versions but with different price
modified = (
    snap_v2.join(snap_v5, on="product_id")
    .filter(F.col("v2.price") != F.col("v5.price"))
    .select(
        F.col("product_id"),
        F.col("v2.name").alias("name"),
        F.col("v2.price").alias("price_v2"),
        F.col("v5.price").alias("price_v5"),
        F.lit("PRICE CHANGED").alias("change")
    )
)

print("Rows added between v2 and v5:")
display(added.select("product_id", "name", "price", "change"))

print("\nRows removed between v2 and v5:")
display(removed.select("product_id", "name", "price", "change"))

print("\nRows with price changes between v2 and v5:")
display(modified.orderBy("product_id"))

## 5c — Find Which Commit Introduced a Specific Change

Binary-search through `history()` to identify the exact version where a value changed.

In [ ]:
# Find the first version where product_id=1 price dropped below 1200
dt = DeltaTable.forName(spark, TABLE)
max_version = dt.history(1).collect()[0]["version"]

print("Searching for the commit that first set price < 1200 for product_id = 1:")
for v in range(max_version + 1):
    rows = spark.sql(
        f"SELECT price FROM {TABLE} VERSION AS OF {v} WHERE product_id = 1"
    ).collect()
    if rows:
        price = rows[0]["price"]
        op    = dt.history().filter(f"version = {v}").collect()[0]["operation"]
        marker = "  ← first version below 1200" if price < 1200 else ""
        print(f"  v{v}: price = {price:>8.2f}  ({op}){marker}")

# Part 6 — `RESTORE TABLE` In Depth

`RESTORE TABLE` reverts the table to a prior version in-place.  
Crucially, it does this by **adding a new commit** to the log — not by rewinding it.  
The bad version remains visible in `history()`, preserving the full audit trail.

```
Before RESTORE:  v0  v1  v2  v3(bad)          ← current = v3
After RESTORE:   v0  v1  v2  v3(bad)  v4(RESTORE TO v2)  ← current = v4 (data = v2)
```

RESTORE is equivalent to: read v2's active file list → write those files as `add` actions in a new commit → write the old v3 files as `remove` actions.

In [ ]:
# Confirm the problem: version 6 zeroed all Electronics prices
print("Current state — Electronics prices are 0 (from the accidental v6 UPDATE):")
display(
    spark.table(TABLE)
    .filter("category = 'Electronics'")
    .orderBy("product_id")
    .select("product_id", "name", "price")
)

In [ ]:
# Identify the last good version (v5 — before the accidental update)
dt = DeltaTable.forName(spark, TABLE)
display(
    dt.history()
    .select("version", "timestamp", "operation", "operationParameters")
    .orderBy("version")
)

In [ ]:
# RESTORE to version 5 (last good state)
# Can also use: RESTORE TABLE {TABLE} TO TIMESTAMP AS OF '...'
spark.sql(f"RESTORE TABLE {TABLE} TO VERSION AS OF 5")

print("After RESTORE TO VERSION AS OF 5:")
display(
    spark.table(TABLE)
    .filter("category = 'Electronics'")
    .orderBy("product_id")
    .select("product_id", "name", "price")
)
# Prices should be back to their correct values

In [ ]:
# RESTORE adds a new version — the bad commit is still visible in history
dt = DeltaTable.forName(spark, TABLE)
display(
    dt.history()
    .select("version", "timestamp", "operation", "operationParameters", "operationMetrics")
    .orderBy("version")
)
# The RESTORE entry shows: operation = 'RESTORE', operationParameters includes the target version

### `RESTORE` `operationMetrics`

| Metric | Meaning |
|--------|---------|
| `numRestoredFiles` | Parquet files re-activated from the target version |
| `numRemovedFiles` | Parquet files from the current (bad) version now logically removed |
| `removedFilesSize` | Total bytes of the removed files |
| `numOfDataFilesRestored` | Data files (not log files) that were restored |

In [ ]:
dt = DeltaTable.forName(spark, TABLE)
restore_entry = dt.history().filter("operation = 'RESTORE'").collect()[0]

print("RESTORE operationMetrics:")
for k, v in (restore_entry["operationMetrics"] or {}).items():
    print(f"  {k:30s}: {v}")

### Selective Restore — Recover One Column Without Full Rollback

Sometimes you only want to restore the values of one column, not roll back the entire table.  
Read the historical snapshot and join it back against the current table as a targeted UPDATE.

In [ ]:
# Simulate: accidentally zeroed prices again on Electronics only
dt = DeltaTable.forName(spark, TABLE)
dt.update("category = 'Electronics'", {"price": F.lit(0.0)})

print("After second accidental zero:")
display(spark.table(TABLE).filter("category = 'Electronics'").select("product_id", "name", "price"))

In [ ]:
# Recover only the price column from the last good version — keep everything else current
dt = DeltaTable.forName(spark, TABLE)
current_version = dt.history(1).collect()[0]["version"]
good_version    = current_version - 1   # one version back

# Read the good prices from the prior version
good_prices = (
    spark.sql(f"SELECT product_id, price FROM {TABLE} VERSION AS OF {good_version}")
    .filter("price > 0")   # only bring back rows that had a real price
)

# Apply those prices back via MERGE — only touches price, leaves everything else intact
dt = DeltaTable.forName(spark, TABLE)
(
    dt.merge(good_prices, "target.product_id = source.product_id")
    .whenMatchedUpdate(set={"price": "source.price"})
    .execute()
)

print("After selective price restore via MERGE:")
display(spark.table(TABLE).filter("category = 'Electronics'").select("product_id", "name", "price"))

# Part 7 — Time Travel Boundaries

Time travel is only possible while the underlying Parquet files still exist on storage.  
Once `VACUUM` removes them, those versions are permanently inaccessible.

### What controls the time travel window

```
─────────────────────────────────────────────────────────► time
│ Table created                                          │ Now
│                                                        │
│ ◄─── delta.deletedFileRetentionDuration (7d) ────────►│
│       ↑ time travel works within this window
│
│ ◄──────── delta.logRetentionDuration (30d) ──────────►│
│       ↑ history() shows versions within this window
```

- **Data files** are kept for 7 days (`deletedFileRetentionDuration`) — this limits time travel.
- **Log files** are kept for 30 days (`logRetentionDuration`) — `history()` shows further back than time travel works.

You can read history entries for a version even after its data files are vacuumed — but you cannot read the data.

In [ ]:
# Check the current retention settings on the table
dt = DeltaTable.forName(spark, TABLE)
props = dt.detail().select("properties").collect()[0]["properties"] or {}

data_retention = props.get("delta.deletedFileRetentionDuration", "interval 7 days  (default)")
log_retention  = props.get("delta.logRetentionDuration",         "interval 30 days (default)")

print(f"delta.deletedFileRetentionDuration : {data_retention}")
print(f"delta.logRetentionDuration         : {log_retention}")
print()
print("To extend time travel to 30 days, run:")
print(f"  ALTER TABLE {TABLE}")
print(f"  SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 30 days')")

In [ ]:
# Attempting to time travel past the VACUUM boundary raises an AnalysisException.
# With fresh tables all files are within the 7-day window so no error here.
# The error message you would see is:
#
#   AnalysisException: The Delta version X has been vacuumed and is no longer available.
#   Please specify a version >= Y or a timestamp >= Z.
#
# To find the earliest version still accessible after VACUUM, check:

earliest = spark.sql(f"""
    SELECT MIN(version) AS earliest_version
    FROM   (DESCRIBE HISTORY {TABLE})
""").collect()[0]["earliest_version"]

print(f"Earliest version visible in history: {earliest}")
print("(All versions shown in history are accessible for time travel in this demo")
print(" because no VACUUM has been run yet and all files are < 7 days old.)")

# Summary

## `history()` / `DESCRIBE HISTORY`

```python
# Python API — returns a DataFrame, newest-first
DeltaTable.forName(spark, table).history()        # all versions
DeltaTable.forName(spark, table).history(10)      # last 10 versions

# SQL — supports LIMIT
spark.sql(f"DESCRIBE HISTORY {table} LIMIT 5")
```

## Time Travel Syntax Reference

| Method | DataFrame API | SQL |
|--------|--------------|-----|
| By version | `.option("versionAsOf", 3)` | `VERSION AS OF 3` / `table@v3` |
| By timestamp | `.option("timestampAsOf", "2025-01-15 09:00:00")` | `TIMESTAMP AS OF '2025-01-15 09:00:00'` |
| Short path form | `spark.read.format("delta").load("path@v3")` | — |

## `RESTORE TABLE`

```sql
-- By version
RESTORE TABLE catalog.schema.table TO VERSION AS OF 5;

-- By timestamp
RESTORE TABLE catalog.schema.table TO TIMESTAMP AS OF '2025-03-20 09:00:00';
```
- Adds a **new commit** — the bad version remains in `history()`.
- `operationMetrics` shows `numRestoredFiles` and `numRemovedFiles`.

## Time Travel Window

| Property | Default | Controls |
|----------|---------|----------|
| `delta.deletedFileRetentionDuration` | 7 days | How far back time travel works (data files) |
| `delta.logRetentionDuration` | 30 days | How far back `history()` shows entries (log files) |

## Practical Patterns

| Goal | Technique |
|------|----------|
| Full rollback | `RESTORE TABLE … TO VERSION AS OF n` |
| Recover one column | Read old version → MERGE back to current |
| Audit a row's changes | Union all `VERSION AS OF v` with a `_version` column |
| Diff two versions | `left_anti` join on primary key + column comparison |
| Find when a value changed | Loop versions + compare target column |